# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['WBPMTFLFDM', 'AAUIKZRTWY'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[23,  2, 16, 13, 20,  6, 12,  6,  4, 13],
       [ 1,  1, 21,  9, 11, 26, 18, 20, 23, 25]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 13,  4,  6, 12,  6, 20, 13, 16,  2],
       [ 0, 25, 23, 20, 18, 26, 11,  9, 21,  1]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[13,  4,  6, 12,  6, 20, 13, 16,  2, 23],
       [25, 23, 20, 18, 26, 11,  9, 21,  1,  1]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        #self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
        #                                   return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        self.decoder_rnn = tf.keras.layers.RNN(
            self.decoder_cell,
            return_sequences=True,
            return_state=True
        )

        self.dense_combine = tf.keras.layers.Dense(self.hidden)
        
    #@tf.function
    def call(self, enc_ids, dec_ids):
        
        # 1. 编码
        enc_emb = self.embed_layer(enc_ids) 
        enc_out, enc_state = self.encoder(enc_emb) # enc_state: (batch, hidden)
        
        # 2. 准备解码
        dec_emb = self.embed_layer(dec_ids) # (batch, seq_len, emb_sz)
        
        batch_size = tf.shape(enc_ids)[0]
        max_len = tf.shape(dec_ids)[1]
        
        # 初始化 TensorArray 用于存储每一步的 logits
        # 数据类型 float32, 动态大小
        logits_ta = tf.TensorArray(dtype=tf.float32, size=0, dynamic_size=True)
        
        # 初始化解码器状态
        # 确保它是张量，不是列表
        current_state = enc_state
        
        # --- 使用 tf.range 进行迭代 ---
        # 这比 tf.while_loop 更容易处理，因为它不需要手动管理循环变量的类型一致性
        for t in tf.range(max_len):
            # 1. 获取当前时间步的输入
            # dec_emb[:, t, :] -> (batch, emb_sz)
            current_input = dec_emb[:, t, :]
            # RNN 需要 (batch, time_steps, emb_sz)
            current_input = tf.expand_dims(current_input, axis=1)
            
            # 2. 运行 RNN 单步
            # 注意：SimpleRNN 的 state 是 (batch, hidden)
            rnn_output, new_state = self.decoder_rnn(current_input, initial_state=[current_state])
            
            # 提取隐藏状态 h_t -> (batch, hidden)
            dec_h = rnn_output[:, 0, :]
            
            # 3. 注意力机制 (Bilinear)
            # dec_h: (batch, hidden) -> (batch, 1, hidden)
            h_for_attn = self.dense_attn(dec_h)
            h_expanded = tf.expand_dims(h_for_attn, axis=1)
            
            # enc_out: (batch, enc_len, hidden) -> (batch, hidden, enc_len)
            enc_out_trans = tf.transpose(enc_out, perm=[0, 2, 1])
            
            # 计算分数: (batch, 1, hidden) * (batch, hidden, enc_len) -> (batch, 1, enc_len)
            attention_scores = tf.matmul(h_expanded, enc_out_trans)
            
            # 计算权重
            attention_weights = tf.nn.softmax(attention_scores, axis=-1)
            
            # 计算上下文向量: (batch, 1, enc_len) * (batch, enc_len, hidden) -> (batch, 1, hidden)
            context = tf.matmul(attention_weights, enc_out)
            context = tf.squeeze(context, axis=1) # -> (batch, hidden)
            
            # 4. 融合并预测
            # 拼接 [context, dec_h]
            combined = tf.concat([context, dec_h], axis=-1)
            output_h = self.dense_combine(combined)
            logits_step = self.dense(output_h) # (batch, v_sz)
            
            # 写入 TensorArray
            logits_ta = logits_ta.write(t, logits_step)
            
            # 更新状态
            current_state = new_state[0]
            
        # 5. 整理输出
        # stack 后形状: (max_len, batch, v_sz)
        logits = logits_ta.stack()
        # 转置为: (batch, max_len, v_sz) 以匹配 loss 函数的要求
        logits = tf.transpose(logits, perm=[1, 0, 2])
        tf.print()
        return logits
    
    
    #@tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, enc_state
    
    def get_next_token(self, x, state, enc_out):
        '''
        推理时单步解码
        shape(x) = [b_sz,] 
        shape(state) = [b_sz, hidden]
        '''
        # 1. 嵌入
        x_emb = self.embed_layer(x) 
        x_emb = tf.expand_dims(x_emb, axis=1) 
        
        # 2. RNN 单步
        # 注意：SimpleRNN 的 state 直接传入即可
        dec_out_step, new_state = self.decoder_rnn(x_emb, initial_state=[state])
        dec_h = dec_out_step[:, 0, :] 
        
        # 3. 注意力
        dec_h_for_attn = self.dense_attn(dec_h) 
        attention_scores = tf.matmul(tf.expand_dims(dec_h_for_attn, axis=1), 
                                     tf.transpose(enc_out, perm=[0, 2, 1])) 
        attention_weights = tf.nn.softmax(attention_scores, axis=-1) 
        context = tf.matmul(attention_weights, enc_out) 
        context = context[:, 0, :] 
        
        # 4. 输出
        combined = tf.concat([context, dec_h], axis=-1) 
        combined = self.dense_combine(combined) 
        logits = self.dense(combined) 
        
        cur_token = tf.argmax(logits, axis=-1)  # 取最大概率 token
        return cur_token, new_state[0]  # 返回 token + 正确状态

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

#@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)


step 0 : loss 3.3059478




















































































































































































































































































































































































































































































































step 500 : loss 0.4270621



































































































































































































































































































































































































































































<tf.Tensor: shape=(), dtype=float32, numpy=0.05772379785776138>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state)[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, 20, enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, True, False, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, False, True, False, False, True, True, True, True, False, True, True, False, True, False]
[('UJRJVLVHZUNWOYCKZMIF', 'FIMZKCYOWNUZHVLVJRJU'), ('DTDJRGOIYXAAATUZEUOC', 'UCOHEZUTAAXYIOGRJDTD'), ('LHQNKJBWGSRVHFMENCEB', 'BECNEMFHVRSGWBJKNQHL'), ('WNTLLRVWLLQHWBSIVFJD', 'DJFVISBWHQLLWVRLLTNW'), ('HMLDLGQQVASCETRSEYSS', 'SSYESRTECSAVQQGLDLMH'), ('DAVKTTBXVGCVGUKSOJIH', 'HIJOSKUGVCGVXBTTKVAD'), ('MEWSPBGRYDBNTVLFUYDP', 'PDYUFLVTNBDYRGBPSWEM'), ('NQBIVPUWBDDNFVTRMIFU', 'UFIMRTVFNDDBWUPVIBQN'), ('WUBJYGMDTAXVFNCAVOXR', 'RXOVACNFVXATDMGYJBUW'), ('DWWFCYCZGIJDVWSOMUQQ', 'QQUMOSWVDJIGZCYCFWWD'), ('BEFHSHZPDGNPFCPWGHTK', 'RKTHGWPCFPNGDPZHSFEB'), ('HPDPMOKGLCVMRJPIJEHS', 'SHEJIPJRMVCLGKOMPDPH'), ('OGTHCRRZEPFBADCHSKDB', 'BDKSHCDABFPEZRRCHTGO'), ('JYKICEBPCXSTUCOFVNHQ', 'QHNVFOCUTSXCPBECIKYJ'), ('WVMPBZCILLGESIRFOMVS', 'SVMOFRISEGLLICZBPMVW'), ('QJNWCQJIPMEXETLDFXUT', 'TUXFDLTEXEMPIJQCWNJQ